# Lesson 6a: Modern Architectures and Transfer Learning — Theory

5a/5b's CNN was two convolutional layers deep. Real vision networks are
tens to hundreds of layers deep, and naively stacking more of 5a's plain
conv-ReLU blocks runs into exactly the vanishing-gradient problem 3a
diagnosed for MLPs — worse, because a 2015 finding (the *degradation
problem*, not simple overfitting) showed a plain 56-layer network
performing measurably worse on training error than its 20-layer
counterpart, purely from optimisation difficulty. This notebook derives
three ideas that let networks scale to real depth and real efficiency —
**residual connections**, **normalisation placement**, and
**depthwise-separable convolution** — and closes with a map of the named
architecture families that introduced each idea historically.

By the end of this notebook you will have:
- derived, and confirmed at extreme depth, why a **residual connection**
  turns a multiplicative gradient chain into an additive one, curing the
  vanishing-gradient problem that depth alone makes worse,
- compared **pre-activation and post-activation** normalisation placement
  and shown which one keeps the residual path's gradient truly unimpeded,
- derived **depthwise-separable convolution**'s parameter saving with
  exact arithmetic, verified against PyTorch's `groups` parameter, and
- summarised the **named architecture families** (AlexNet through Vision
  Transformer) and what each one specifically changed.

## Introduction

3a showed that a sufficiently deep plain network's per-layer gradient
norm decays geometrically, because backpropagation through $L$ stacked
layers multiplies by $L$ Jacobian-like factors. Making a network *deeper*
without changing anything else makes this multiplicative chain longer,
not shorter — so naively stacking more of 5a's conv-ReLU blocks should
make training *harder*, not just slower, past some depth. This is exactly
what was observed empirically before residual connections existed: adding
more plain layers eventually made even *training* error worse, not just
test error, ruling out overfitting as the explanation. Every idea below
addresses a different piece of the same underlying problem — depth should
help a network represent more, but only if the gradient can still reach
the early layers to make use of it.

## Setup

In [ ]:
# Fixed seeds: every stochastic step in this notebook (weight init, random
# vectors) is reproducible.
import numpy as np
import torch

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

import matplotlib.pyplot as plt
import torch.nn as nn

plt.rcParams["figure.figsize"] = (5, 4)
print("numpy:", np.__version__)
print("torch:", torch.__version__)


def relu(z):
    return np.maximum(0.0, z)


def relu_prime_z(z):
    return (z > 0.0).astype(np.float64)

## Residual Connections

A plain block computes $a_{l} = \text{ReLU}(W_l a_{l-1})$, so
backpropagating the loss gradient through $L$ such blocks multiplies by
$L$ factors of $W_l^\top \odot \text{ReLU}'(z_l)$ — exactly 3a's vanishing
chain, now applied at whatever depth the architecture chooses. A
**residual block** instead computes

$$a_l = a_{l-1} + F_l(a_{l-1}), \qquad F_l(a_{l-1}) = \text{ReLU}(W_l a_{l-1}),$$

and differentiating the sum directly (not the product) gives

$$\frac{\partial a_l}{\partial a_{l-1}} = I + \frac{\partial F_l}{\partial a_{l-1}}.$$

The gradient back through $L$ residual blocks is a product of $L$ terms
that are each "identity plus something", which — critically — is an
**additive** combination of paths once expanded, not a chain of $L$
multiplied Jacobians that must all stay near 1 to avoid vanishing. Even if
every single $\partial F_l/\partial a_{l-1}$ is small (exactly the
vanishing-gradient regime 3a demonstrated), the identity term guarantees
at least one path of gradient magnitude 1 all the way back to any earlier
layer — a "gradient highway" that a plain stack simply does not have.

In [ ]:
def init_block(width, seed):
    rng = np.random.default_rng(seed)
    return rng.normal(0.0, 0.6 * np.sqrt(2.0 / width), size=(width, width))


def forward_stack(Ws, x, residual):
    activations, zs = [x], []
    a = x
    for W in Ws:
        z = W @ a
        zs.append(z)
        a = (a + relu(z)) if residual else relu(z)
        activations.append(a)
    return activations, zs


def backward_grad_norms(Ws, activations, zs, dL_dy, residual):
    delta = dL_dy
    norms = []
    for l in range(len(Ws) - 1, -1, -1):
        local = delta * relu_prime_z(zs[l])
        grad_W = local @ activations[l].T
        norms.append(np.linalg.norm(grad_W))
        branch = Ws[l].T @ local
        delta = (delta + branch) if residual else branch
    norms.reverse()
    return norms


WIDTH, BATCH = 64, 32
rng = np.random.default_rng(SEED)
x0 = rng.normal(size=(WIDTH, BATCH))
# A fixed regression target -- independent of network depth or weight scale -- so that
# dL_dy = (output - target) genuinely reflects how well each depth's network can still be
# corrected by gradient descent, rather than trivially rescaling with the network itself.
y_target = rng.normal(size=(WIDTH, BATCH)) * 0.1

depths = [5, 10, 20, 30, 40, 50, 60, 80, 100]
first_block_norms = {False: [], True: []}
for depth in depths:
    Ws = [init_block(WIDTH, seed=l) for l in range(depth)]
    for residual in (False, True):
        activations, zs = forward_stack(Ws, x0, residual)
        dL_dy = (activations[-1] - y_target) / BATCH
        norms = backward_grad_norms(Ws, activations, zs, dL_dy, residual)
        first_block_norms[residual].append(norms[0])  # gradient reaching the earliest block

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(depths, first_block_norms[False], marker="o", label="plain stack")
ax.plot(depths, first_block_norms[True], marker="o", label="residual stack")
ax.set_yscale("log")
ax.set_xlabel("total network depth")
ax.set_ylabel("gradient norm reaching the earliest block (log scale)")
ax.set_title("Does the earliest layer still get a usable gradient, as depth grows?")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

print(f"depth {depths[-1]}: plain stack earliest-block gradient    = {first_block_norms[False][-1]:.3e}")
print(f"depth {depths[-1]}: residual stack earliest-block gradient = {first_block_norms[True][-1]:.3e}")
assert first_block_norms[True][-1] > first_block_norms[False][-1] * 1e6

At shallow depth, both stacks pass a usable gradient back to the
earliest block. As depth grows, the plain stack's gradient collapses
smoothly and monotonically toward numerical insignificance — exactly 3a's
vanishing-gradient argument, now driven by depth alone rather than by
activation choice or initialisation scale. The residual stack's gradient
does not vanish; if anything it grows with depth, since the identity term
means every block's contribution *adds* to the signal reaching the input
rather than being multiplied away. (That unchecked growth is itself worth
noting — an unnormalised residual stream can grow large purely from
repeated addition, which is exactly the problem normalisation placement,
covered next, exists to temper.) The two curves are separated by tens of
orders of magnitude by depth 100: the plain stack has, for practical
purposes, stopped receiving any training signal at its earliest layers.

## Normalisation Placement

The original ResNet block placed its final non-linearity **after** the
residual addition: $a_l = \text{ReLU}(a_{l-1} + F_l(a_{l-1}))$
("post-activation"). He et al.'s follow-up "Identity Mappings" paper
(2016) found that moving normalisation and the activation **inside** the
branch, before the addition, trains more reliably at very large depth:
$a_l = a_{l-1} + F_l(\text{Norm}(a_{l-1}))$ ("pre-activation"), with
**no non-linearity ever applied to the sum itself**.

The difference matters for exactly the gradient-highway argument above.
Post-activation's final $\text{ReLU}$ sits on the path both the identity
*and* the branch must pass through before reaching the next block, so the
identity term derived above picks up an extra $\text{ReLU}'(\cdot)$ factor
at every block boundary — a factor that can be exactly 0 for some units,
locally breaking the highway. Pre-activation keeps the addition itself
completely free of any non-linearity: the identity term at every block is
*exactly* 1, with the norm and activation confined entirely to the
branch, where they were always intended to act on the *transformation*,
not on the *shortcut*.

In [ ]:
def layernorm_forward(z, eps=1e-5):
    mu = z.mean(axis=0, keepdims=True)
    var = z.var(axis=0, keepdims=True)
    y = (z - mu) / np.sqrt(var + eps)
    return y, (y, var, eps)


def layernorm_backward(dy, cache):
    y, var, eps = cache
    std_inv = 1.0 / np.sqrt(var + eps)
    return std_inv * (dy - dy.mean(axis=0, keepdims=True) - y * (dy * y).mean(axis=0, keepdims=True))


def post_act_forward(Ws, x):
    caches = []
    a = x
    for W in Ws:
        z = W @ a
        y_ln, ln_cache = layernorm_forward(z)
        h = relu(y_ln)
        y = a + h
        a_next = relu(y)
        caches.append((a, ln_cache, y_ln, y))
        a = a_next
    return a, caches


def post_act_backward(caches, Ws, dL_dy):
    delta = dL_dy
    norms = []
    for l in range(len(Ws) - 1, -1, -1):
        a_prev, ln_cache, y_ln, y = caches[l]
        delta = delta * relu_prime_z(y)              # through the final ReLU (hits identity AND branch)
        d_yln = delta * relu_prime_z(y_ln)            # through ReLU inside the branch
        dz = layernorm_backward(d_yln, ln_cache)
        grad_W = dz @ a_prev.T
        norms.append(np.linalg.norm(grad_W))
        delta = delta + Ws[l].T @ dz                  # identity (already ReLU'd above) + branch
    norms.reverse()
    return norms


def pre_act_forward(Ws, x):
    caches = []
    a = x
    for W in Ws:
        y_ln, ln_cache = layernorm_forward(a)
        h = relu(y_ln)
        z = W @ h
        a_next = a + z                                # no non-linearity on the sum
        caches.append((a, ln_cache, y_ln, h))
        a = a_next
    return a, caches


def pre_act_backward(caches, Ws, dL_dy):
    delta = dL_dy
    norms = []
    for l in range(len(Ws) - 1, -1, -1):
        a_prev, ln_cache, y_ln, h = caches[l]
        d_identity = delta                            # untouched -- no ReLU ever sits on the sum
        grad_W = delta @ h.T
        norms.append(np.linalg.norm(grad_W))
        dh = Ws[l].T @ delta
        d_yln = dh * relu_prime_z(y_ln)
        d_branch = layernorm_backward(d_yln, ln_cache)
        delta = d_identity + d_branch
    norms.reverse()
    return norms


norm_depths = [10, 30, 60, 100, 200, 400]
first_block_norm_placement = {"post": [], "pre": []}
dL_dy_fixed = rng.normal(size=(WIDTH, BATCH)) * 0.1
for depth in norm_depths:
    Ws_norm = [init_block(WIDTH, seed=l) for l in range(depth)]
    _, post_caches = post_act_forward(Ws_norm, x0)
    post_norms = post_act_backward(post_caches, Ws_norm, dL_dy_fixed)
    _, pre_caches = pre_act_forward(Ws_norm, x0)
    pre_norms = pre_act_backward(pre_caches, Ws_norm, dL_dy_fixed)
    first_block_norm_placement["post"].append(post_norms[0])
    first_block_norm_placement["pre"].append(pre_norms[0])

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(norm_depths, first_block_norm_placement["post"], marker="o", label="post-activation (ReLU after the sum)")
ax.plot(norm_depths, first_block_norm_placement["pre"], marker="o", label="pre-activation (no non-linearity on the sum)")
ax.set_yscale("log")
ax.set_xlabel("total network depth")
ax.set_ylabel("gradient norm reaching the earliest block (log scale)")
ax.set_title("Post- vs. pre-activation, as depth grows")
ax.legend(fontsize=8)
ax.grid(alpha=0.3)
plt.show()

for depth, post_n, pre_n in zip(norm_depths, first_block_norm_placement["post"], first_block_norm_placement["pre"]):
    print(f"depth {depth:4d}: post={post_n:.3e}  pre={pre_n:.3e}  ratio(pre/post)={pre_n / post_n:.2f}")
assert first_block_norm_placement["pre"][-1] > first_block_norm_placement["post"][-1]

Post-activation's earliest-block gradient stays roughly flat as depth
grows — the per-block ReLU-after-the-sum factor caps how much signal can
accumulate at the shortcut, regardless of how many more blocks are added.
Pre-activation's earliest-block gradient keeps *growing* with depth, the
same unimpeded-accumulation behaviour "Residual Connections" showed for
an ordinary residual stack, because pre-activation never places a
non-linearity on the sum itself. The ratio between the two widens steadily
with depth — pre-activation's advantage over post-activation is not a
fixed constant, it compounds the deeper the network gets, which is
exactly why the literature's largest residual networks (100+ layers)
adopted pre-activation specifically.

## Efficient Convolutions

5a's convolution costs $C_{in}\cdot C_{out}\cdot k^2$ parameters: every
output channel gets its own $k\times k$ kernel over *every* input channel.
**Depthwise-separable convolution** (used throughout MobileNet) factors
this into two much cheaper steps: a **depthwise** convolution applies one
$k\times k$ kernel *per input channel independently* (no mixing across
channels at all — $C_{in}\cdot k^2$ parameters), followed by a
**pointwise** ($1\times1$) convolution that mixes channels
($C_{in}\cdot C_{out}$ parameters). The combined parameter count is

$$C_{in}k^2 + C_{in}C_{out} \quad\text{vs.}\quad C_{in}C_{out}k^2,$$

a ratio of $\frac{1}{C_{out}} + \frac{1}{k^2}$ — for a typical
$3\times3$ kernel with many output channels, close to $\frac19$, an
almost order-of-magnitude reduction with no change to the spatial
receptive field of a single block.

In [ ]:
C_in, C_out, k = 64, 128, 3

standard_params = C_in * C_out * k * k
depthwise_params = C_in * k * k
pointwise_params = C_in * C_out
separable_params = depthwise_params + pointwise_params

print(f"standard conv:        {standard_params:,} parameters")
print(f"depthwise ({C_in} groups): {depthwise_params:,} parameters")
print(f"pointwise (1x1):      {pointwise_params:,} parameters")
print(f"depthwise-separable:  {separable_params:,} parameters")
print(f"reduction factor: {standard_params / separable_params:.1f}x "
      f"(formula: {1/C_out + 1/(k*k):.3f} vs. measured ratio {separable_params/standard_params:.3f})")

# Verify the parameter counts against real PyTorch layers, not just the formula.
standard_conv = nn.Conv2d(C_in, C_out, kernel_size=k, bias=False)
depthwise_conv = nn.Conv2d(C_in, C_in, kernel_size=k, groups=C_in, bias=False)
pointwise_conv = nn.Conv2d(C_in, C_out, kernel_size=1, bias=False)

standard_actual = sum(p.numel() for p in standard_conv.parameters())
separable_actual = sum(p.numel() for p in depthwise_conv.parameters()) + \
    sum(p.numel() for p in pointwise_conv.parameters())
print(f"\ntorch standard nn.Conv2d parameters:  {standard_actual:,}")
print(f"torch depthwise+pointwise parameters: {separable_actual:,}")
assert standard_actual == standard_params
assert separable_actual == separable_params

x = torch.randn(1, C_in, 16, 16)
out_separable = pointwise_conv(depthwise_conv(x))
print(f"\ndepthwise-separable output shape: {tuple(out_separable.shape)} "
      f"(same spatial size and channel count as a standard {C_out}-channel conv would produce)")

The formula, a direct parameter count, and PyTorch's actual layer sizes
all agree: swapping one standard $3\times3$ convolution for a
depthwise-separable pair keeps the same input/output channel counts and
spatial behaviour while using roughly a ninth of the weights — the
efficiency argument behind every mobile-oriented CNN architecture that
followed MobileNet.

## Architecture Families

Every named vision architecture below changed exactly one or two things
relative to what came before it; none of them required reinventing
convolution itself.

| Architecture | Year | Key idea | What it changed |
|---|---|---|---|
| AlexNet | 2012 | ReLU + dropout + GPU training at scale | First deep CNN to beat hand-engineered features decisively on ImageNet |
| VGG | 2014 | Uniform stacks of small $3\times3$ convolutions | Showed depth from small, repeated kernels beats a few large ones (5a's receptive-field argument, stacked deliberately) |
| GoogLeNet / Inception | 2014 | Parallel multi-scale branches, $1\times1$ convolutions for channel reduction | Let one layer capture features at several receptive-field sizes at once, cheaply |
| ResNet | 2015 | Residual connections | Made networks over 100 layers deep actually trainable ("Residual Connections" above) |
| Pre-activation ResNet | 2016 | Norm and activation moved inside the branch | Kept the residual shortcut a truly unimpeded identity path ("Normalisation Placement" above) |
| MobileNet | 2017 | Depthwise-separable convolution | Cut convolution cost roughly 8-9x for mobile/edge deployment ("Efficient Convolutions" above) |
| EfficientNet | 2019 | Compound scaling of depth, width and input resolution together | Replaced ad-hoc "make it bigger" with a principled joint scaling rule |
| Vision Transformer (ViT) | 2020 | Image patches + self-attention, no convolution at all | Replaced local connectivity and parameter sharing (5a's whole argument for convolution) with the attention mechanism 9a-10a derive |

Every architecture in this table except the last keeps 5a's core
commitment to convolution; each one instead changes how deep, how
normalised, or how efficient that convolution can be made. The Vision
Transformer is the outlier worth flagging now: it drops convolution's
inductive bias entirely and lets attention (9a) learn which pixels relate
to which from data — a strategy that only became practical once training
sets grew large enough to substitute for the bias convolution had
supplied by construction.

## Key Takeaways

- A **residual connection** ($a_l = a_{l-1} + F_l(a_{l-1})$) turns the
  backward pass from a multiplicative chain of $L$ Jacobians into an
  additive combination of paths, guaranteeing a gradient-magnitude-1
  "highway" back to any earlier layer regardless of how small any
  individual branch's gradient becomes — confirmed directly by comparing
  per-block gradient norms of a 60-block plain stack against an
  identically-sized residual stack.
- **Pre-activation placement** (norm and activation confined entirely
  inside the branch, never on the residual sum itself) keeps that
  identity term exactly 1 at every block; **post-activation** placement
  (the original ResNet's final ReLU after the sum) puts an extra
  ReLU-derivative factor on both the shortcut and the branch at every
  block boundary, measurably degrading gradient flow at the same depth.
- **Depthwise-separable convolution** factors a standard convolution's
  $C_{in}C_{out}k^2$ parameters into a depthwise step ($C_{in}k^2$) and a
  pointwise step ($C_{in}C_{out}$), a combined cost verified against
  PyTorch's `groups` parameter to be roughly a ninth of the standard
  convolution's for typical channel counts.
- The named architecture families each changed one or two specific
  things — depth (VGG), multi-scale branches (Inception), residual
  connections (ResNet), normalisation placement (pre-activation ResNet),
  efficiency (MobileNet, EfficientNet) — right up to the Vision
  Transformer, which replaces convolution's inductive bias with the
  attention mechanism this series derives next.